# NLP Group 7 Project P2

## Imports

In [5]:
from datasets import load_dataset
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os

/Users/fannar/miniconda3/envs/nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load dataset

In [6]:
dataset = load_dataset("MathArena/final_answer_comps", split="train")
df = dataset.to_pandas()

## General info and null values

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   problem_idx   139 non-null    int64 
 1   answer        139 non-null    object
 2   problem_type  130 non-null    object
 3   problem       139 non-null    object
 4   competition   139 non-null    object
 5   source        9 non-null      object
dtypes: int64(1), object(5)
memory usage: 6.6+ KB


In [8]:
print("Dataset Head:\n", df.head(), sep="", end="\n" + "="*50 + "\n")
print("Dataset Columns:\n", df.columns, sep="", end="\n" + "="*50 + "\n")
print("Dataset Shape:\n", df.shape, sep="", end="\n" + "="*50 + "\n")

Dataset Head:
   problem_idx answer                    problem_type  \
0            1     70                 [Number Theory]   
1            2    588                      [Geometry]   
2            3     16                 [Combinatorics]   
3            4    117                       [Algebra]   
4            5    279  [Combinatorics, Number Theory]   

                                             problem     competition source  
0  Find the sum of all integer bases $b>9$ for wh...  aime/aime_2025   None  
1  On $\triangle ABC$ points $A, D, E$, and $B$ l...  aime/aime_2025   None  
2  The 9 members of a baseball team went to an ic...  aime/aime_2025   None  
3  Find the number of ordered pairs $(x,y)$, wher...  aime/aime_2025   None  
4  There are $8!= 40320$ eight-digit positive int...  aime/aime_2025   None  
Dataset Columns:
Index(['problem_idx', 'answer', 'problem_type', 'problem', 'competition',
       'source'],
      dtype='object')
Dataset Shape:
(139, 6)


## Communication with the model

In [9]:
load_dotenv()

ENDPOINT = "https://nlp-pcaf.services.ai.azure.com/openai/v1/"
MODEL_NAME = "DeepSeek-V3-0324"
DEPLOYMENT_NAME = "DeepSeek-V3-0324"

api_key = os.getenv("API_KEY")

client = OpenAI(
    base_url=f"{ENDPOINT}",
    api_key=api_key
)

completion = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is **Paris**. It is one of the most famous and visited cities in the world, known for landmarks like the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral.  \n\nWould you like information on anything specific about Paris or France? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [10]:
def get_llm_response(messages,  temperature=0.0):
    """Sends a request to the Azure LLM with a System Role Prompt."""
    # Configure the response format for JSON
    response_format = {"type": "text"}
    if "JSON" in messages[0] or "JSON" in messages[1]:
         response_format = {"type": "json_object"}
    
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            temperature=temperature,
            response_format=response_format
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM API Error: {e}"

## Role Prompts

In [11]:
VERIFIER_JSON_SCHEMA = """
{
  "valid": <boolean: true if final answer is correct, false otherwise>,
  "error_category": <string: one of 'CALCULATION_RISK', 'CONCEPTUAL_FLAW', 'LOGIC_OMISSION', 'NONE'>,
  "critique_summary": <string: a brief, actionable explanation of the error>
}
"""

VERIFIER_ROLE = (
    "You are the VERIFIER, a hyper-critical mathematical auditor. "
    "Your goal is to FIND FLAWS. Assume the Solver's solution is INCORRECT until proven otherwise.\n\n"
    "## CHECKLIST\n"
    "1. Does the code output actually match the final text answer?\n"
    "2. Did the Solver handle edge cases (e.g., n=0, n=1)?\n"
    "3. [MANDATORY] Geometric Hallucination: If the problem involves polygons/areas and no (x,y) coordinates are used, REJECT as CONCEPTUAL_FLAW.\n"
    "4. [MANDATORY] Combinatorial Explosion: If the answer is found via manual listing (>10 items) instead of code/formula, REJECT as CALCULATION_RISK.\n"
    "5. [MANDATORY] Symbolic Math: If roots/equations are simplified manually without `sympy`, REJECT as CALCULATION_RISK.\n\n"
    "6. [MANDATORY] Pseudo-Recall: If derivation relies on 'I recall' or unproven formulas, flag as LOGIC_OMISSION.\n\n"
    "If you are unsure, reject it with a request for verification code. "
    f"Output strictly valid JSON only matching this schema: {VERIFIER_JSON_SCHEMA}"
)

# Example Problem
PROBLEM = (
    "A rectangular garden has sides in the ratio 4:3. If the area of the garden is 300 m^2, "
    "what is the length of the fence needed to enclose it? Provide your final answer as an integer."
)

# Example of a Flawed Solution (Solver's first attempt for PoC)
# This simulates the Solver making a calculation error: 2*(20+15) = 70, but the Solver will output 90.
FLAWED_SOLUTION_S1 = """
Solution:
1. Let the length be 4x and the width be 3x.
2. Area: (4x)(3x) = 12x^2.
3. 12x^2 = 300, so x^2 = 25, and x = 5.
4. Sides are 4(5)=20m and 3(5)=15m.
5. Perimeter P = 2(20 + 15) = 2(45) = 90m.
Final Answer: 90
"""

In [12]:
import sys
import io
import contextlib
from func_timeout import func_timeout, FunctionTimedOut

def execute_python_code(code_str, timeout_seconds=5):
    """
    Executes Python code with a strict timeout.
    Returns the output or a Timeout Error message.
    """
    output_buffer = io.StringIO()
    
    # Pre-import common libraries
    exec_globals = {
        "__builtins__": __builtins__,
        "print": print
    }
    
    try:
        import math
        import sympy
        import numpy as np
        exec_globals.update({"math": math, "sympy": sympy, "np": np})
        
        # Define the actual execution logic as a wrapper
        def _run_captured():
            with contextlib.redirect_stdout(output_buffer):
                exec(code_str, exec_globals)
        
        # Run with timeout
        func_timeout(timeout_seconds, _run_captured)
        
        return output_buffer.getvalue()
        
    except FunctionTimedOut:
        return f"RUNTIME ERROR: Code execution exceeded {timeout_seconds} seconds. Likely infinite loop."
        
    except Exception as e:
        return f"RUNTIME ERROR: {str(e)}"

# --- Unit Test the Sandbox ---
test_code = """
import sympy
x = sympy.symbols('x')
sol = sympy.solve(x**2 - 4, x)
print(sol)
while True:
    pass
"""
print(f"Sandbox Test Output: {execute_python_code(test_code)}")

Sandbox Test Output: RUNTIME ERROR: Code execution exceeded 5 seconds. Likely infinite loop.


In [13]:
import re

def run_solver_with_tools(system_prompt, user_problem, max_turns=5):
    """
    Manages the 'Code Interpreter' loop for the Solver.
    Detects ```python blocks, runs them, and feeds output back to the model.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_problem}
    ]
    
    # Track the full conversation for this problem attempt
    full_trace = ""
    
    for turn in range(max_turns):
        # 1. Get LLM Response
        response = get_llm_response(messages)
        content = response
        full_trace += f"\n\n--- Step {turn+1} ---\n{content}"
        
        # 2. Check for Code Blocks (Greedy match: finds the last python block)
        # Regex looks for ```python ... ```
        code_match = re.search(r"```python\n(.*?)```", content, re.DOTALL)
        
        if code_match:
            code_str = code_match.group(1)
            print(f"    [Tool Use] Executing Python Code...")
            
            # 3. Execute Code
            execution_output = execute_python_code(code_str)
            print(f"    [Tool Output] {execution_output.strip()[:100]}...") # Log brief output
            
            # 4. Append Result to History so Model sees it
            messages.append({"role": "assistant", "content": content})
            messages.append({
                "role": "user", 
                "content": f"OBSERVATION (Code Output):\n{execution_output}\n\nContinue reasoning."
            })
            
            full_trace += f"\n\n[OBSERVATION]:\n{execution_output}"
            
        else:
            # No code found implies the model is done or just chatting. 
            # If it has "Final Answer" or boxed, we stop.
            if "\\boxed" in content or "Final Answer" in content:
                return content, full_trace
            
            # If no code and no answer, we force one more turn to encourage conclusion
            messages.append({"role": "assistant", "content": content})
            if turn == max_turns - 1:
                break
                
    return content, full_trace

In [14]:
def get_solver_prompt(few_shot_examples=None, is_retry=False):
    """
    Constructs the System Prompt for the Solver Agent (Tool-Enabled).
    """
    base_prompt = (
        "You are the SOLVER, an expert mathematician equipped with a Python Code Interpreter.\n\n"
        "## TOOL USE PROTOCOL\n"
        "1. To calculate, count, or solve equations, write code inside a ```python ... ``` block.\n"
        "2. The system will execute it and return the output as an 'OBSERVATION'.\n"
        "3. Read the OBSERVATION and continue your reasoning.\n"
        "4. You can use `sympy`, `math`, and `numpy`.\n"
        "5. ALWAYS print() the result you need to see. If you don't print, you won't see it.\n\n"
        "## MANDATORY CONSTRAINTS\n"
        "1. The Coordinate Mandate: For geometry, define (x,y) coordinates and use Python to calculate areas/distances.\n"
        "2. The Code-First Mandate: For counting >10 items, write a Python script to iterate.\n"
        "3. The Symbolic Math Mandate: For roots/equations, use `sympy` in your code block.\n"
        "## CODING BEST PRACTICES\n"
        "- When printing lists, slice them (e.g., `print(my_list[:20])`) to avoid output truncation.\n"
        "- Import all necessary libraries (`math`, `sympy`, `numpy`, `itertools`) inside your code block.\n"
        "- If you get a RUNTIME ERROR, read the error message carefully and fix the bug in the next turn.\n"
    )

    if few_shot_examples:
        base_prompt += f"\n## REFERENCE EXAMPLES\n{few_shot_examples}\n"

    if is_retry:
        base_prompt += (
            "\n**CORRECTION MODE**: Your previous answer was rejected. "
            "Follow the PLANNER'S instruction strictly."
        )

    return base_prompt 

# FOR BASELINE (Current State):
SOLVER_ROLE = get_solver_prompt(few_shot_examples=None)

# FOR FUTURE PCAF (Future State):
# few_shots = "... content from MathInstruct ..."
# SOLVER_ROLE_PCAF = get_solver_prompt(few_shot_examples=few_shots)

In [15]:
# --- PLANNER LOGIC: DETERMINISTIC ROUTING ---

def construct_planner_feedback(verifier_json, problem_text):
    """
    Parses Verifier JSON and constructs the 'Planner Hint' for the next Solver iteration.
    Routes 'CALCULATION_RISK' to either Iteration or SymPy based on context.
    """
    error_cat = verifier_json.get("error_category", "UNKNOWN")
    critique = verifier_json.get("critique_summary", "")
    
    # Base feedback
    instruction = f"The Verifier identified a {error_cat}. Feedback: {critique}."
    
    # 1. GEOMETRY ROUTE
    if error_cat == "CONCEPTUAL_FLAW" and any(k in problem_text.lower() for k in ["geometry", "triangle", "polygon", "area"]):
        instruction += "\n\n**MANDATORY ACTION:** RE-SOLVE using Coordinate Geometry. Assign (0,0) to a vertex and calculate using the Shoelace Formula."
        
    # 2. ALGEBRA/CALCULATION ROUTES
    elif error_cat == "CALCULATION_RISK":
        # Check for Algebra keywords
        if any(k in problem_text.lower() or k in critique.lower() for k in ["root", "equation", "quadratic", "solve", "system"]):
            instruction += (
                "\n\n**MANDATORY ACTION:** RE-SOLVE using Python (sympy). "
                "Define variables as symbols (e.g., `x, y = symbols('x y')`) and use `solve()` to handle the algebra precisely."
            )
        # Check for Counting keywords
        elif any(k in problem_text.lower() or k in critique.lower() for k in ["count", "list", "iterate", "permutations"]):
            instruction += (
                "\n\n**MANDATORY ACTION:** RE-SOLVE using Python Logic. "
                "Write a script to iterate through the valid cases explicitly to avoid double-counting."
            )
        else:
            # Fallback for generic calculation errors
            instruction += "\n\n**ACTION:** Verify your arithmetic using a Python code block."
            
    return instruction

In [16]:
import re

def extract_answer(text):
    """
    Extracts the final answer from LLM output.
    Priority 1: Look for \boxed{...} (Standard math format)
    Priority 2: Look for 'Final Answer: X' pattern
    Priority 3: Fallback to the very last number found in the text.
    """
    if not isinstance(text, str):
        return None

    # 1. Check for \boxed{answer} (Most reliable, used in your PoC)
    boxed_match = re.search(r'\\boxed\{([^}]+)\}', text)
    if boxed_match:
        return boxed_match.group(1).strip()

    # 2. Check for "Final Answer: <number>" pattern
    final_answer_match = re.search(r'(?:Final Answer|answer is)[:\s]*([-\d\.]+)', text, re.IGNORECASE)
    if final_answer_match:
        return final_answer_match.group(1).strip()

    # 3. Fallback: Find all numbers and return the last one
    # This handles cases where the model just ends with the number
    numbers = re.findall(r'[-+]?\d*\.\d+|\d+', text)
    if numbers:
        return numbers[-1]
        
    return None


In [17]:
def check_correctness(prediction, ground_truth):
    """
    Compares the extracted prediction with the ground truth.
    Handles type mismatches (string vs int) and float formatting.
    """
    if prediction is None or ground_truth is None:
        return False
        
    # Normalize to strings first
    pred_str = str(prediction).strip()
    gt_str = str(ground_truth).strip()
    
    # 1. Direct String Match
    if pred_str == gt_str:
        return True
        
    # 2. Numerical Match (Handles 70.0 vs 70)
    try:
        pred_float = float(pred_str)
        gt_float = float(gt_str)
        # Use a small epsilon for float comparison
        return abs(pred_float - gt_float) < 1e-6
    except ValueError:
        pass
        
    return False

In [18]:
import json

def extract_json_from_response(text):
    """
    Robustly extracts the first valid JSON object from a string, 
    ignoring Markdown fences, preamble text, or trailing comments.
    """
    try:
        # 1. Fast path: try parsing the raw string first
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. Regex Strategy: Find everything between the first '{' and the last '}'
    # re.DOTALL allows '.' to match newlines
    match = re.search(r"(\{.*\})", text, re.DOTALL)
    if match:
        candidate = match.group(1)
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            # Sometimes models leave trailing commas (e.g., {"a":1,}) which breaks valid JSON
            # Simple cleanup for trailing commas before closing braces
            candidate = re.sub(r",\s*\}", "}", candidate)
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass
    
    # 3. Last Resort: Aggressive Markdown Stripping
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None  # Failed completely

In [19]:

def run_pcaf_on_problem(problem_text, max_retries=3):
    all_tries = []

    # 1. Initial Attempt (Solver with Tools)
    print(f"\n--- Turn 0: Initial Solver Attempt (Tool-Enabled) ---")
    
    # CALL THE NEW INNER LOOP HERE
    current_solution, trace = run_solver_with_tools(
        get_solver_prompt(is_retry=False), 
        problem_text
    )

    current_try = {
        "turn": 0,
        "solution_text": current_solution,
        "full_trace": trace,
        "verifier_json": None, # Will be filled if we verify
        "planner_instruction": None
    }
    all_tries.append(current_try)
    
    for i in range(1, max_retries + 1):
        # 2. Verification Step
        print(f"--- Turn {i}: Verifying... ---")
        verifier_input = f"Problem: {problem_text}\n\nProposed Solution Trace:\n{trace}"

        messages = [
        {"role": "system", "content": VERIFIER_ROLE},
        {"role": "user", "content": verifier_input}
    ]
        
        verifier_raw = get_llm_response(messages)
        verifier_json = extract_json_from_response(verifier_raw)

        if verifier_json is None:
            # Fallback: If JSON fails, force a generic "Continue" or stop
            print(f"System Error: Verifier produced invalid JSON. Raw output: {verifier_raw[:50]}...")
            all_tries[-1]["error"] = "Verifier JSON Parse Fail"
            # Optional: Break here, or treat as 'Valid' to avoid crashing. 
            # For debugging, breaking is safer.
            break
        
        all_tries[-1]["verifier_json"] = verifier_json
        
        is_valid = verifier_json.get("valid", False)
        print(f"Verifier Verdict: {'VALID' if is_valid else 'INVALID'} ({verifier_json.get('error_category')})")
            
        if is_valid:
            return current_solution, all_tries
                
        # 3. Planning Step 
        planner_instruction = construct_planner_feedback(verifier_json, problem_text)
        print(f"Planner Instruction: {planner_instruction}")
            
        # 4. Correction Step (Solver Retry WITH TOOLS)
        print(f"--- Turn {i}: Solver Correction ---")
        solver_input_context = (
            f"ORIGINAL PROBLEM: {problem_text}\n\n"
            f"PREVIOUS ATTEMPT TRACE:\n{trace}\n\n"
            f"CRITIQUE & INSTRUCTION:\n{planner_instruction}\n\n"
            "Generate the corrected solution now. USE PYTHON to verify your fix."
        )
            
        # CALL THE INNER LOOP HERE AGAIN
        current_solution, new_trace = run_solver_with_tools(
            get_solver_prompt(is_retry=True), 
            solver_input_context
        )
        trace += f"\n\n--- Correction {i} ---\n{new_trace}"

        # Store this new attempt
        new_try = {
            "turn": i,
            "solution_text": current_solution,
            "full_trace": new_trace,
            "verifier_json": None,
            "planner_instruction": None
        }
        all_tries.append(new_try)
            
            
    return current_solution, all_tries

## Test Case
The following code block is used to test the functionality of the above two functions (extract_answers and check_correctness)

In [20]:
# Test on your current dataframe head
print("--- Evaluation Pipeline Sanity Check ---")

# Let's test against the first few rows of your loaded dataframe
for index, row in df.head().iterrows():
    ground_truth = row['answer']
    
    # Simulate a "Perfect" LLM response using the \boxed{} format you saw in the PoC
    simulated_llm_output = f"After calculating, the answer is \\boxed{{{ground_truth}}}"
    
    # Run extraction
    extracted = extract_answer(simulated_llm_output)
    
    # Run correctness check
    is_correct = check_correctness(extracted, ground_truth)
    
    print(f"Prob ID {row['problem_idx']}: GT={ground_truth} | Extracted={extracted} | Correct? {is_correct}")

# Test a tricky failure case
print("\n--- Edge Case Test ---")
tricky_gt = 70
tricky_output = "The calculation gives 69.999 which rounds to 70. Final Answer: 70."
ext = extract_answer(tricky_output)
print(f"GT={tricky_gt} | Output='...Final Answer: 70.' | Extracted={ext} | Correct? {check_correctness(ext, tricky_gt)}")

--- Evaluation Pipeline Sanity Check ---
Prob ID 1: GT=70 | Extracted=70 | Correct? True
Prob ID 2: GT=588 | Extracted=588 | Correct? True
Prob ID 3: GT=16 | Extracted=16 | Correct? True
Prob ID 4: GT=117 | Extracted=117 | Correct? True
Prob ID 5: GT=279 | Extracted=279 | Correct? True

--- Edge Case Test ---
GT=70 | Output='...Final Answer: 70.' | Extracted=70. | Correct? True


# CoT Baseline
Here we generate the Chain-of-Thought baseline for N samples.

In [21]:
import pandas as pd
import time

def run_baseline_evaluation(dataframe, num_samples=5):
    """
    Runs the Zero-Shot CoT Baseline (Solver only) on a subset of the dataframe.
    """
    results = []
    
    # 1. Select the subset (first N rows)
    # Using .copy() to ensure we don't accidentally modify the original slice
    subset = dataframe.head(num_samples).copy()
    
    print(f"--- Starting Baseline Run on {num_samples} problems ---")
    
    for index, row in subset.iterrows():
        problem_id = row['problem_idx']
        problem_text = row['problem']
        ground_truth = row['answer']
        
        print(f"Processing Problem ID: {problem_id}...", end=" ")
        
        # 2. Get Solver Response (Zero-Shot CoT)
        # Using the function and role you defined in main.ipynb
        try:
            llm_output = get_llm_response(SOLVER_ROLE, problem_text, temperature=0.0)
        except Exception as e:
            llm_output = f"ERROR: {str(e)}"
            print("API Fail")
        
        # 3. Extract and Score
        extracted_val = extract_answer(llm_output)
        is_correct = check_correctness(extracted_val, ground_truth)
        
        # Log to console for real-time tracking
        status = "PASS" if is_correct else "FAIL"
        print(f"{status} | GT: {ground_truth} | Pred: {extracted_val}")
        
        # 4. Record Data
        results.append({
            "problem_idx": problem_id,
            "problem_type": row['problem_type'],
            "ground_truth": ground_truth,
            "extracted_answer": extracted_val,
            "is_correct": is_correct,
            "llm_output": llm_output  # Keep full text for error analysis later
        })
        
        # Optional: Sleep briefly to avoid hitting tight rate limits if needed
        # time.sleep(0.5) 

    # 5. Compile Results
    results_df = pd.DataFrame(results)
    
    # Calculate Accuracy
    accuracy = results_df['is_correct'].mean() * 100
    print(f"\n--- Baseline Complete ---")
    print(f"Accuracy: {accuracy:.2f}% ({results_df['is_correct'].sum()}/{num_samples})")
    
    return results_df

# --- EXECUTION ---
# Run the baseline on 5 samples
#baseline_results = run_baseline_evaluation(df, num_samples=5)

# Save to CSV for your records (as per Project Timeline Week 1)
#baseline_results.to_csv("baseline_results_cutoff.csv", index=False)
#print("\nResults saved to 'baseline_results_cutoff.csv'")

# Display the failure cases (to understand what the Verifier needs to catch)
#print("\n--- Failure Analysis (Incorrect Rows) ---")
#failures = baseline_results[~baseline_results['is_correct']]
#if not failures.empty:
#    print(failures[['problem_idx', 'ground_truth', 'extracted_answer']])
#else:
#    print("No failures in this small batch!")

RUN WITH THE ITERATIVE LOOP LOGIC MAX-TRY = 3

In [22]:
import pandas as pd
import json

# --- START PCAF EVALUATION ---

# 1. Select the first 5 problems from the loaded DataFrame 'df'
test_subset = df.head(30) 
pcaf_results = []

print("=======================================================")
print(f"STARTING PCAF TEST ON FIRST {len(test_subset)} PROBLEMS (N=3 ITERATIONS MAX)")
print("=======================================================")

# 2. Loop over the test subset
for index, row in test_subset.iterrows():
    problem_id = row['problem_idx']
    problem_text = row['problem']
    problem_type = row['problem_type']
    ground_truth = str(row['answer']) 

    print(f"\n--- RUNNING PCAF FOR PROBLEM {index + 1}/{len(test_subset)} (ID: {problem_id}) ---")
    
    # 3. Call the main PCAF loop function
    final_output, history = run_pcaf_on_problem(problem_text, max_retries=3)
    
    # 4. Analyze the results from the final attempt
    final_attempt = history[-1]
    final_solver_output = final_attempt['solution_text']
    
    final_extracted_answer = extract_answer(final_solver_output) 
    is_correct = check_correctness(final_extracted_answer, ground_truth)
    
    # 5. Summarize and store
    result_summary = {
        'problem_idx': problem_id,
        'problem_type': problem_type,
        'ground_truth': ground_truth,
        'pcaf_answer': final_extracted_answer,
        'is_correct': is_correct,
        'attempts_used': len(history),
        'final_status': 'Verified' if is_correct else 'Failed',
        'final_verifier_valid': final_attempt['verifier_json'].get('valid', False) if final_attempt['verifier_json'] else 'N/A',
        # --- NEW LOGGING FIELD ---
        'full_history_json': json.dumps(history, indent=2) # Convert the history list to a readable JSON string
        # -------------------------
    }
    pcaf_results.append(result_summary)
    
    # Log summary for quick review during the run
    print(f"\nRESULTS: Correct? {'YES' if is_correct else 'NO'} | Attempts: {len(history)}")

print("\n=======================================================")
print("PCAF Test Complete. Generating Results DataFrame.")

# 6. Convert results to DataFrame and display summary
pcaf_results_df = pd.DataFrame(pcaf_results)

accuracy = pcaf_results_df['is_correct'].mean() * 100
print(f"\n--- Baseline Complete ---")
print(f"Accuracy: {accuracy:.2f}% ({pcaf_results_df['is_correct'].sum()}/{30})")

print("\nFirst 5 PCAF Results Summary (including full history):")
print(pcaf_results_df[['problem_idx', 'is_correct', 'attempts_used', 'pcaf_answer']])

# Save the results to a CSV file for documentation
pcaf_results_df.to_csv('pcaf_30_test_results_detailed.csv', index=False) 
print("\nDetailed results saved to pcaf_5_test_results_detailed.csv")

STARTING PCAF TEST ON FIRST 30 PROBLEMS (N=3 ITERATIONS MAX)

--- RUNNING PCAF FOR PROBLEM 1/30 (ID: 1) ---

--- Turn 0: Initial Solver Attempt (Tool-Enabled) ---
    [Tool Use] Executing Python Code...
    [Tool Output] RUNTIME ERROR: f-string: single '}' is not allowed (<string>, line 11)...
    [Tool Use] Executing Python Code...
    [Tool Output] Base 21 works: 17_b = 28, 97_b = 196, division = 7.0
Sum of valid bases: 21...
    [Tool Use] Executing Python Code...
    [Tool Output] Base 21 works: 17_b = 28, 97_b = 196, division = 7.0
Valid bases: [21]
Sum of valid bases: 21...
    [Tool Use] Executing Python Code...
    [Tool Output] RUNTIME ERROR: int() base must be >= 2 and <= 36, or 0...
    [Tool Use] Executing Python Code...
    [Tool Output] Base 21 works: 17_b = 28, 97_b = 196, division = 7.0
Base 49 works: 17_b = 56, 97_b = 448, division ...
--- Turn 1: Verifying... ---
Verifier Verdict: VALID (NONE)

RESULTS: Correct? YES | Attempts: 1

--- RUNNING PCAF FOR PROBLEM 2/30 (ID

# Observed failure modes from the above baseline evaluation:

## CRITIQUE RUBRIC: KNOWN FAILURE MODES
You must audit the Solver's solution for the following specific "Red Flags" that historically cause failure. If found, reject the solution with the corresponding Error Category.

1.  **Geometric Hallucination (Red Flag: Visual Intuition)**
    * **Trigger:** The problem involves polygons, complex areas, or points inside/outside shapes (especially "heptagons", "nested triangles", or "reflected points").
    * **Check:** Did the Solver assign explicit $(x,y)$ coordinates to vertices?
    * **Failure Mode:** If the Solver used "visual subtraction" (e.g., "Area = Big Triangle - Small Triangle") without proving the points lie strictly inside, or if it made assumptions about shape regularity.
    * **Action:** If coordinates are missing, flag as `CONCEPTUAL_FLAW`.
    * **Feedback:** "Solution relies on visual intuition. You must use Coordinate Geometry (Shoelace Formula) to guarantee the points are correctly located."

2.  **Combinatorial Explosion (Red Flag: Manual Enumeration)**
    * **Trigger:** Counting problems (Combinatorics, Permutations, Number Theory) where the answer is likely > 50.
    * **Check:** Did the Solver attempt to manually list or iterate cases in the text (e.g., "Case 1...", "Case 2...")?
    * **Failure Mode:** Manual listing of more than 10 items consistently leads to off-by-one errors or double-counting (as seen in divisibility problems).
    * **Action:** If manual listing > 10 items is found, flag as `CALCULATION_RISK`.
    * **Feedback:** "Manual enumeration of large sets is prone to error. You must write a Python script to iterate and count these cases precisely."

3.  **Pseudo-Recall (Red Flag: "I recall...")**
    * **Trigger:** Phrases like "I recall the answer is...", "It is known that...", or "Using the standard formula for X..." (without deriving it).
    * **Check:** Does the solution derivation rely on an unproven "recalled" fact?
    * **Action:** Flag as `LOGIC_OMISSION`.
    * **Feedback:** "Do not rely on recalled answers or obscure formulas. Derive the result from first principles or use Python to verify the formula."

4.  **Algebraic Complexity (Red Flag: Messy Roots)**
    * **Trigger:** The final answer contains unsimplified square roots (e.g., $\sqrt{47}$), complex fractions, or decimals when the problem implies an integer answer (e.g., "Find the number of...", "Find the sum of...").
    * **Check:** Did the Solver manually simplify a quadratic equation or a system of linear equations in the text?
    * **Failure Mode:** LLMs notoriously fail at simplifying expressions like $\frac{23\sqrt{5}-6}{5}$ into integers. They also frequently make sign errors in manual substitution.
    * **Action:** If manual symbolic manipulation is found for complex roots, flag as `CALCULATION_RISK`.
    * **Feedback:** "Potential arithmetic error in simplifying roots. Do not simplify manually. Use Python's `sympy` library to solve the equation symbolically and obtain the exact integer."

## PoC run

In [23]:
# MatchArena's AIME first problem
dummy_problem = df.iloc[0][3]
#solver_output = get_llm_response(SOLVER_ROLE, dummy_problem, temperature=0.0)
print(solver_output)

/var/folders/jq/kb6cwcjx28q_jsdt_29r26b80000gn/T/ipykernel_52549/3975429595.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dummy_problem = df.iloc[0][3]


NameError: name 'solver_output' is not defined

In [ ]:
problem_solution_pairs = {
    dummy_problem: solver_output,
    PROBLEM: FLAWED_SOLUTION_S1
}

In [ ]:
for problem, solution in problem_solution_pairs.items():
    break

    verifier_input = (
        f"Problem: {problem}\n\n"
        f"Solution to Critique:\n{solution}"
    )

    verifier_raw_output = get_llm_response(VERIFIER_ROLE, verifier_input, temperature=0.0)
    verifier_raw_output = verifier_raw_output.replace("```json", "")
    verifier_raw_output = verifier_raw_output.replace("```", "")
    verifier_raw_output = verifier_raw_output.strip()

    try:
        # JSON parsing
        verifier_json = json.loads(verifier_raw_output)
        print("JSON Parsing Successful. Verifier Output:")
        print(json.dumps(verifier_json, indent=2))
        
        # Check if the critique is valid and get correction signal
        if verifier_json.get('valid') == False:
            error_cat = verifier_json.get('error_category', 'UNKNOWN_ERROR')
            critique = verifier_json.get('critique_summary', 'No summary provided.')
            
            print("\nSolution is invalid. Preparing Planner's Correction...")
            
            # Planner logic
            planner_correction_hint = (
                f"CRITIQUE: The Verifier identified a **{error_cat}** at the final step. "
                f"Specifically: **{critique}**. You must rigorously re-examine your final calculation."
            )

            # Next Solver input
            solver_input_s2 = (
                f"ORIGINAL PROBLEM: {problem}\n\n"
                f"PREVIOUS FAILED ATTEMPT:\n{solution}\n\n"
                f"PLANNER'S CORRECTION HINT:\n{planner_correction_hint}\n\n"
                f"--- GENERATE CORRECTED SOLUTION (ATTEMPT S2) ---"
            )
            
        else:
            print("\nSolution passed. Loop terminates.")
            solver_input_s2 = None
            
    except json.JSONDecodeError as e:
        print(f"JSON Failure: Error: {e}")
        solver_input_s2 = None # Fail the loop

    # PCAF iteration 2 (Solver Correction)
    if solver_input_s2:
        print("\n--- 2. SOLVER CORRECTION (Targeted Generation Proof) ---")
        
        # Send the corrected prompt to the single LLM instance
        solver_corrected_output = get_llm_response(SOLVER_ROLE, solver_input_s2, temperature=0.0)
        
        print("Solver's Corrected Solution (S2):")
        print(solver_corrected_output)
        
        # We can run the verifier again and have multiple iterations but this is for PoC.
        print("\n[PoC Complete] The system successfully ran one full corrective loop.")
        print("The final correctness of S2 would be checked against the MathArena gold standard.")

JSON Parsing Successful. Verifier Output:
{
  "valid": true,
  "error_category": "NONE",
  "critique_summary": "The solution is correct and well-reasoned, with all steps logically sound and calculations accurate."
}

Solution passed. Loop terminates.
JSON Parsing Successful. Verifier Output:
{
  "valid": true,
  "error_category": "NONE",
  "critique_summary": "The solution correctly follows the perimeter calculation steps, with accurate calculations and logical flow."
}

Solution passed. Loop terminates.
